In [ ]:
# !pip install jieba
# !pip install evaluate

In [15]:
import jieba
import evaluate


In [16]:
# English Example
predictions = ["hello, I don't understand."]
references = [
    ["hello, I don't know."]
]
bleu = evaluate.load("bleu")
results = bleu.compute(predictions=predictions, references=references)
print(results)

{'bleu': 0.537284965911771, 'precisions': [0.8333333333333334, 0.6, 0.5, 0.3333333333333333], 'brevity_penalty': 1.0, 'length_ratio': 1.0, 'translation_length': 6, 'reference_length': 6}


In [17]:
# Chinese Example
space_sent = "因 此 ， 我 们 知 道 ， 洪 水 是 气 候 和 气 候 变 化 的 结 果 ， 不 同 的 发 生 在 不 同 的 情 况 下 发 生 了 不 同 的 。"
sent = ''
for word in space_sent:
    if word != " ":
        sent += word
print(sent)
words = list(jieba.cut(sent, cut_all=False)) # tokeize sentence using jieba
sent_pred = ''
for word in words:
    if sent_pred == '':
        sent_pred += word
    else:
        sent_pred += ' ' + word
print(sent_pred)

print()

sent = "洪水的产生是气候和河道共同作用的结果，不同河道形态下洪水产生的特点是不同的。"
print(sent)
words = list(jieba.cut(sent, cut_all=False)) # tokeize sentence using jieba
sent_ref = ''
for word in words:
    if sent_ref == '':
        sent_ref += word
    else:
        sent_ref += ' ' + word
print(sent_ref)

print()

predictions = [sent_pred]
references = [
    [sent_ref]
]
bleu = evaluate.load("bleu")
results = bleu.compute(predictions=predictions, references=references)
print(results)

因此，我们知道，洪水是气候和气候变化的结果，不同的发生在不同的情况下发生了不同的。
因此 ， 我们 知道 ， 洪水 是 气候 和 气候变化 的 结果 ， 不同 的 发生 在 不同 的 情况 下 发生 了 不同 的 。

洪水的产生是气候和河道共同作用的结果，不同河道形态下洪水产生的特点是不同的。
洪水 的 产生 是 气候 和 河道 共同 作用 的 结果 ， 不同 河道 形态 下 洪水 产生 的 特点 是 不同 的 。

{'bleu': 0.18180608220159192, 'precisions': [0.5384615384615384, 0.28, 0.16666666666666666, 0.043478260869565216], 'brevity_penalty': 1.0, 'length_ratio': 1.0833333333333333, 'translation_length': 26, 'reference_length': 24}


In [ ]:
## reference code
# word segmentation on .txt file
# each line of this file includes one sentence
import jieba
import sys

input_file = sys.argv[1] # input text file
output_file = sys.argv[2] # output file to save results

with open(input_file,'r') as f2:
    sents = f2.readlines() # read lines of input file

lengths = []
with open(output_file,'w') as f1: # to save outputs
    for sent in sents: # one sentence at a time
        # f1.write(sent.strip()+'\t-->\t') # dump original sentence
        words = list(jieba.cut(sent, cut_all=False)) # tokeize sentence using jieba
        lengths.append(len(words)) # keep record of sentence lengths
        for word in words:
            if word == '\n':
                # f1.write(word.encode('utf-8'))
                f1.write(word)
            else:
            	# f1.write(word.encode('utf-8') + ' ') # dump tokenized sentence with tab separation
                f1.write(word + ' ')  
            # pdb.set_trace()


print('Minimum length: ',min(lengths))
print('Maximum length: ',max(lengths))
print('Average length: ',sum(lengths)/len(lengths))

## 3. Evaluation of Trained Transformer Model on Validation/Test Set

Below we implement the evaluation script that loads our trained Transformer model (`save/models/model.pt`), generates Chinese translations for the evaluation set using greedy search (`greedy_decode`), segments both predictions and references using `jieba`, and computes the final BLEU score.

In [ ]:
# 1. Imports and setup
import torch
import jieba
import evaluate
import numpy as np
from model.transformer import build_transformer
from tokenization import PrepareData, MaskBatch
from translator_en2cn import get_config, get_model, greedy_decode

# 2. Config and Data Prep
config = get_config(debug=True) # Set to True for train_mini, False for full dataset
PAD = 0
UNK = 1
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Evaluating using device: {device}")

# Preprocessing data
data = PrepareData(config['train_file'], config['dev_file'], config['batch_size'], UNK, PAD)
src_vocab_size = len(data.en_word_dict)
tgt_vocab_size = len(data.cn_word_dict)

# 3. Load Model and Trained Weights
model = get_model(config, src_vocab_size, tgt_vocab_size).to(device)
try:
    model.load_state_dict(torch.load(config['save_file'], map_location=device))
    print(f"Successfully loaded trained weights from {config['save_file']}")
except Exception as e:
    print(f"Could not load weights: {e}\n(Make sure to train the model first by running python translator_en2cn.py!)")

# 4. Generate Predictions and Calculate BLEU
bleu = evaluate.load("bleu")
predictions = []
references = []

print("Evaluating on validation dataset...")
model.eval()
with torch.no_grad():
    for i, batch in enumerate(data.dev_data):
        encoder_input = batch.src.to(device)
        encoder_mask = batch.src_mask.to(device)
        
        # greedy decode
        model_out = greedy_decode(model, encoder_input, encoder_mask, data.cn_word_dict, config['seq_len'], device)
        
        # reconstruct prediction string
        model_out_text = []
        for j in range(1, model_out.size(0)):
            sym = data.cn_index_dict[model_out[j].item()]
            if sym != 'EOS':
                model_out_text.append(sym)
            else:
                break
        pred_sent = "".join(model_out_text)
        
        # reconstruct reference target string
        ref_words = [data.cn_index_dict[w] for w in data.dev_cn[i]]
        ref_words = [w for w in ref_words if w not in ['BOS', 'EOS']]
        ref_sent = "".join(ref_words)
        
        # Segment with jieba
        pred_segmented = " ".join(list(jieba.cut(pred_sent, cut_all=False)))
        ref_segmented = " ".join(list(jieba.cut(ref_sent, cut_all=False)))
        
        predictions.append(pred_segmented)
        references.append([ref_segmented])
        
        # Print first few samples
        if i < 5:
            source_text = " ".join([data.en_index_dict[w] for w in data.dev_en[i]])
            print(f"--- Example {i+1} ---")
            print(f"SOURCE:    {source_text}")
            print(f"TARGET:    {ref_sent}")
            print(f"PREDICTED: {pred_sent}")

# Compute and print BLEU score
if len(predictions) > 0:
    results = bleu.compute(predictions=predictions, references=references)
    print("\n" + "="*50)
    print(f"Dataset: Validation Set (size: {len(predictions)})")
    print(f"BLEU Score: {results['bleu'] * 100:.4f}%")
    print(f"Precisions: {[round(p*100, 2) for p in results['precisions']]}")
    print(f"Brevity Penalty: {results['brevity_penalty']}")
    print("="*50)
